# Damköhler numbers

## Load the simulations

In [ ]:
from functools import partial
import numpy as np
from lucifex.fdm import GridFunctionSeries, NPyConstantSeries
from lucifex.fem import grid_resample, average_grid
from lucifex.plt import (
    plot_colormap, plot_line, save_figure,
    plot_colormap_multifigure, plot_line_multifigure,
)
from lucifex.utils.array_utils import as_index
from lucifex.io import create_dir_path, find_dir_paths
from lucifex.utils.py_utils import FrozenDict
from lucifex.sim import GridSimulationFromNPZ
from crocodil.dns.system_a import SYSTEM_A_REFERENCE
from crocodil.theory.system_a import (
    mass_dissolved_asymptote, mass_capillary_asymptote,
    mass_dissolved_initial, mass_capillary_initial,
)

# Searching for all simulations in the batch
PARAMS_NUMERICAL = FrozenDict(
    c_stabilization=None,
    c_limits=True,
)
DIR_ROOT = create_dir_path(
    PARAMS_NUMERICAL, 
    dir_root='./',
    dir_prefix='data', 
    dir_params=PARAMS_NUMERICAL.keys(), 
)
DIR_FIGS = f'{DIR_ROOT}/figures'

T_STOP = 120.0
sim_dir_paths = find_dir_paths(
    DIR_ROOT, 
    include=f't_stop={T_STOP}_*',
    contains=('CHECKPOINT.h5', 'c.npz'),
)
print('Before parameter selection')
for i in sim_dir_paths: print(i)

# selecting a batch of simulations
PARAM_NAME = 'Da'PARAM_TEX = 'Da'as_int_if_poss = lambda p: int(p) if isinstance(p, float) and float.is_integer(p) else p
PARAMS_BATCH = SYSTEM_A_REFERENCE

simulations = GridSimulationFromNPZ.dict_from_dir_paths(
    PARAM_NAME, 
    sim_dir_paths,
    ('c', 's'),
    ('f', 'mD', 'mC', 'uRMS'),
    PARAMS_BATCH,
    lazy=False,
    sorting=lambda d: dict(sorted(d.items(), reverse=False)),
)

print('After parameter selection')
for i in simulations.values(): print(i.dir_path)

epsilon, zeta0, sr, cr, aspect = PARAMS_BATCH['epsilon', 'zeta0', 'sr', 'cr', 'aspect']
Ly = 1.0
Lx = aspect * Ly
save_figure = partial(
    save_figure, 
    dir_path=DIR_FIGS, 
    prefix=PARAM_NAME, 
    pickle=True,
    file_ext=('svg', 'png'),
)

In [ ]:
function_series = 'c'
for param_value, sim in simulations.items():
    time_series = sim[function_series].time_series
    print(f'{PARAM_NAME} = {param_value}')
    print(f"n_snapshots = {len(time_series)} t_min = {np.min(time_series)}  t_max = {np.max(time_series)}")

## Flux & mass

In [ ]:
HLINES = False
legend_labels = []
f_lines, uRMS_lines = [], []
mC_lines, mD_lines, mC_frac_lines, mD_frac_lines = [], [], [], []

mC_initial = mass_capillary_initial(epsilon, zeta0, sr, Lx, Ly)
mD_initial = mass_dissolved_initial(zeta0, sr, cr, Lx, Ly)
m_initial = mC_initial + mD_initial

mC_asymp = mass_capillary_asymptote(m_initial, epsilon, Lx, Ly)
mD_asymp = mass_dissolved_asymptote(m_initial, epsilon, Lx, Ly)

for param_value, sim in simulations.items():
    legend_labels.append(as_int_if_poss(param_value))
    f, mC, mD, uRMS = sim['f', 'mC', 'mD', 'uRMS']
    fZeta0, fZetaPlus, fZetaMinus = f.split()
    f_lines.append((fZeta0.time_series, [np.sum(i) for i in fZeta0.value_series]))
    uRMS_lines.append((uRMS.time_series, uRMS.value_series))
    mC_lines.append((mC.time_series, mC.value_series))
    mD_lines.append((mD.time_series, mD.value_series))
    mC_frac_lines.append((mC.time_series, [i / (i + j) for i, j in zip(mC.value_series, mD.value_series)]))
    mD_frac_lines.append((mD.time_series, [j / (i + j) for i, j in zip(mC.value_series, mD.value_series)]))
    
    
line_kws = dict(
    cyc='black',
    x_label='$t$',
    legend_labels=legend_labels,
    legend_title=f'${PARAM_TEX}$',
)
hline_kws = dict(
    xmin=0, xmax=120.0, 
    linestyles=(5, (10, 3)), colors='gray', linewidths=0.75,
)

fig, ax = plot_line(f_lines, y_label='$F(y=\zeta_0)$', x_lims=(0, 10.0), **line_kws)
if DIR_FIGS: save_figure('f(t)_early')(fig)

fig, ax = plot_line(f_lines, y_label='$F(y=\zeta_0)$', x_lims=(0, 60.0), **line_kws)
if DIR_FIGS: save_figure('f(t)_late')(fig)

fig, ax = plot_line(uRMS_lines, y_label='$\mathrm{rms}(\mathbf{u})$', **line_kws)
ax.set_yscale('log')
if DIR_FIGS: save_figure('uRMS(t)')(fig)

fig, ax = plot_line(mC_lines, y_label='$m_C$', **line_kws)
if HLINES: ax.hlines([mC_initial, mC_asymp], **hline_kws)
if DIR_FIGS: save_figure('mC(t)')(fig)

fig, ax = plot_line(mD_lines, y_label='$m_D$', **line_kws)
if HLINES: ax.hlines([mD_initial, mD_asymp], **hline_kws)
if DIR_FIGS: save_figure('mD(t)')(fig)

fig, ax = plot_line(mC_frac_lines, y_label='$m_C/(m_C + m_D)$', **line_kws)
if DIR_FIGS: save_figure('mC(t)_fraction')(fig)

fig, ax = plot_line(mD_frac_lines, y_label='$m_D/(m_C + m_D)$', **line_kws)
if DIR_FIGS: save_figure('mD(t)_fraction')(fig)

## Horizontal averages

In [ ]:
slc = slice(0, None, 10)
titles = []
sAvgX_lines, sAvgX_legend_labels = [], []
cAvgX_lines, cAvgX_legend_labels = [], []

for param_value, sim in simulations.items():
    titles.append(f'\n${PARAM_TEX}={as_int_if_poss(param_value)}$')
    s, c = sim['s', 'c']
    sAvgX = GridFunctionSeries(
        [
            average_grid(grid_resample(i, (1, 10)), 'x')
            for i in s.series[slc]
        ], 
        s.time_series[slc], 
        'sAvgX',
    )
    sAvgX_legend_labels.append((min(sAvgX.time_series), max(sAvgX.time_series)))
    sAvgX_lines.append(sAvgX.series)
    cAvgX = GridFunctionSeries(
        [average_grid(use_cache=True)(i, 'x') for i in c.series[slc]], c.time_series[slc], 'cAvgX',
    )
    cAvgX_legend_labels.append((min(cAvgX.time_series), max(cAvgX.time_series)))
    cAvgX_lines.append(cAvgX.series)


multiline_kws = dict(
    cyc='jet',
    title=titles, 
    legend_title='$t$',
    flip=True,
    y_label='$y$',
)

fig, *_  = plot_line_multifigure(n_cols=1, cbars=True)(
    cAvgX_lines, 
    legend_labels=cAvgX_legend_labels,
    x_label='$\langle c\\rangle_x$', 
    y_lims=(0, Ly),
    **multiline_kws,
)
if DIR_FIGS: save_figure('cAvgX(t)')(fig)

fig, *_  = plot_line_multifigure(n_cols=1, cbars=True)(
    sAvgX_lines, 
    legend_labels=sAvgX_legend_labels,
    x_label='$\langle s\\rangle_x$', 
    y_lims=(zeta0 * Ly, Ly),
    **multiline_kws,
)
if DIR_FIGS: save_figure('sAvgX(t)')(fig)

## Subdomain averages

In [ ]:
legend_labels = []
cPlus_lines, sPlus_lines, cMinus_lines = [], [], []

for param_value, sim in simulations.items():
    legend_labels.append(as_int_if_poss(param_value))
    s, c = sim['s', 'c']
    y_axis = c.mesh.y_axis
    zeta0_index = as_index(y_axis, zeta0)
    slcPlus = slice(zeta0_index, None)
    slcMinus = slice(0, zeta0_index)
    cPlus = NPyConstantSeries(
        average_grid(c.series, ('x', 'y'), (':', slcPlus)), c.time_series, 'cPlus',
    )
    sPlus = NPyConstantSeries(
        average_grid(s.series, ('x', 'y'), (':', slcPlus)), s.time_series, 'sPlus',
    )
    cMinus = NPyConstantSeries(
        average_grid(c.series, ('x', 'y'), (':', slcMinus)), c.time_series, 'cMinus',
    )
    cPlus_lines.append((cPlus.time_series, cPlus.value_series))
    sPlus_lines.append((sPlus.time_series, sPlus.value_series))
    cMinus_lines.append((cMinus.time_series, cMinus.value_series))

line_kws = dict(
    cyc='black',
    x_label='$t$',
    legend_labels=legend_labels,
    legend_title=f'${PARAM_TEX}$',
)

fig, ax = plot_line(
    cPlus_lines,
    y_label='$c^+$',
    **line_kws,
)
if DIR_FIGS: save_figure('cPlus(t)')(fig)

fig, ax = plot_line(
    sPlus_lines,
    y_label='$s^+$',
    **line_kws,
)
if DIR_FIGS: save_figure('sPlus(t)')(fig)

fig, ax = plot_line(
    cMinus_lines,
    y_label='$c^-$',
    **line_kws,
)
if DIR_FIGS: save_figure('cMinus(t)')(fig)

## Visualization

In [ ]:
c_t_targets = (5.0, 10.0, 20.0, 100.0)
s_t_targets = c_t_targets

for c_t_trg, s_t_trg in zip(c_t_targets, s_t_targets):
    c_funcs, c_titles = [], []
    s_funcs, s_titles = [], []

    for param_value, sim in simulations.items():
        as_int_if_poss = as_int_if_poss(param_value)
        c = sim['c']
        c_time_index = as_index(c.time_series, c_t_trg)
        c_funcs.append(c.series[c_time_index])
        c_titles.append(f'${PARAM_TEX}={as_int_if_poss}$\n $c(t={c.time_series[c_time_index]:.3f})$')
        s = sim['s']
        s_time_index = as_index(s.time_series, s_t_trg)
        s_funcs.append(grid_resample(use_cache=True)(s.series[s_time_index], (1, 10)))
        s_titles.append(f'${PARAM_TEX}={as_int_if_poss}$\n $s(t={s.time_series[s_time_index]:.3f})$')

    fig, *_  = plot_colormap_multifigure(n_cols=1, cbars=(0, 1))(
        c_funcs, title=c_titles,
    )
    if DIR_FIGS: save_figure(f'c(x,y,t={c_t_trg})')(fig, file_ext=('pdf', 'png'))

    s_cbars = True if PARAM_NAME == 'sr' else (0, sr)
    fig, *_  = plot_colormap_multifigure(n_cols=1, cbars=s_cbars)(
        s_funcs, title=s_titles, y_lims=(zeta0 * Ly, Ly), aspect='auto',
    )
    if DIR_FIGS: save_figure(f's(x,y,t={s_t_trg})')(fig, file_ext=('pdf', 'png'))